# 02 — Préparation & nettoyage des données

Objectif : rendre les données exploitables — déduplication, harmonisation des **NOC historiques**, filtrage des JO d'été et création de la variable cible `Has_Medal`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'config.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Racine projet :', ROOT)

Racine projet : C:\Users\nicol\ynov\Fil Rouge\Projet-Fil-Rouge-J.O


In [2]:
import pandas as pd
from src.data.data_loader import load_raw_data
from src.data.data_cleaner import clean_data, get_medals_df

raw = load_raw_data()
print('Avant nettoyage :', raw.shape)

Avant nettoyage : (252565, 11)


## Problèmes identifiés

Doublons, et plusieurs **codes pays disparus** qu'il faut rattacher à leur équivalent moderne :
GDR/FRG → GER, ROC → RUS, SCG → SRB, BOH → CZE.

In [3]:
print('Doublons :', raw.duplicated().sum())
hist = ['URS', 'GDR', 'FRG', 'ROC', 'SCG', 'BOH']
raw[raw['NOC'].isin(hist)].groupby('NOC').size().to_frame('lignes')

Doublons : 0


,lignes
NOC,
BOH,153
FRG,2558
GDR,2104
ROC,561
SCG,300
URS,4622


## Application du pipeline de nettoyage

In [4]:
clean = clean_data(raw)
print('Après nettoyage :', clean.shape)
print('Saisons :', clean['Season'].unique())
print('Doublons restants :', clean.duplicated().sum())

Après nettoyage : (252565, 12)
Saisons : ['Summer']
Doublons restants : 0


## Vérification des fusions de NOC

In [5]:
for old in ['GDR', 'FRG', 'ROC', 'SCG', 'BOH']:
    print(f'{old} encore présent ? ->', old in clean['NOC'].values)
print('Lignes Allemagne (GER) :', int((clean['NOC'] == 'GER').sum()))

GDR encore présent ? -> False
FRG encore présent ? -> False
ROC encore présent ? -> False
SCG encore présent ? -> False
BOH encore présent ? -> False
Lignes Allemagne (GER) : 13528


## Feature engineering : `Has_Medal`

Variable binaire (1 = médaille remportée) qui servira de cible aux analyses.

In [6]:
print(clean[['Medal', 'Has_Medal']].drop_duplicates().sort_values('Has_Medal').to_string(index=False))
print('\nTaux de lignes médaillées :', round(clean['Has_Medal'].mean() * 100, 2), '%')

   Medal  Has_Medal
No medal          0
    Gold          1
  Bronze          1
  Silver          1

Taux de lignes médaillées : 15.37 %


## Sauvegarde du dataset nettoyé

Le fichier est écrit dans `data/processed/` (ignoré par Git car régénérable).

In [7]:
from config import CLEANED_DATA_FILE
CLEANED_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(CLEANED_DATA_FILE, index=False)
print('Sauvegardé ->', CLEANED_DATA_FILE)

Sauvegardé -> C:\Users\nicol\ynov\Fil Rouge\Projet-Fil-Rouge-J.O\data\processed\olympics_cleaned.csv


## Conclusion

- Doublons supprimés, JO d'été conservés.
- 4 familles de **NOC historiques fusionnées** pour une analyse cohérente des pays.
- Variable `Has_Medal` créée ; dataset prêt pour l'analyse (`03`) et la modélisation (`04`).